# 2WikiMultiHopQA Data Exploration

This notebook validates dataset loading, inspects question distributions, and provides baseline corpus statistics for Phase 1.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from fair_kg_rag.data.dataset_loader import load_dataset

In [ ]:
raw_path = ROOT / 'data' / 'raw' / 'dev.json'
if not raw_path.exists():
    raise FileNotFoundError(
        f'Missing {raw_path}. Run scripts/download_data.py first.'
    )

records = load_dataset(raw_path)
len(records)

In [ ]:
rows = []
for record in records:
    num_paragraphs = len(record.context)
    num_sentences = sum(len(par.sentences) for par in record.context)
    num_supporting_facts = len(record.supporting_facts)
    rows.append({
        'id': record.id,
        'question_type': record.question_type,
        'question_length': len(record.question.split()),
        'num_paragraphs': num_paragraphs,
        'num_sentences': num_sentences,
        'num_supporting_facts': num_supporting_facts,
        'num_evidence_triples': len(record.evidences),
        'num_entity_ids': len(record.wikidata_ids),
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
summary = {
    'num_questions': len(df),
    'avg_question_length': float(df['question_length'].mean()),
    'avg_paragraphs_per_question': float(df['num_paragraphs'].mean()),
    'avg_sentences_per_question': float(df['num_sentences'].mean()),
    'avg_supporting_facts': float(df['num_supporting_facts'].mean()),
    'avg_evidence_triples': float(df['num_evidence_triples'].mean()),
}
summary

In [ ]:
plt.figure(figsize=(8, 4))
order = df['question_type'].value_counts().index
sns.countplot(data=df, x='question_type', order=order)
plt.title('Question Type Distribution')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['num_paragraphs'], bins=20, ax=axes[0])
axes[0].set_title('Paragraphs per Question')

sns.histplot(df['num_supporting_facts'], bins=20, ax=axes[1])
axes[1].set_title('Supporting Facts per Question')

plt.tight_layout()
plt.show()

In [ ]:
sample = records[0]
print('ID:', sample.id)
print('Question:', sample.question)
print('Answer:', sample.answer)
print('Type:', sample.question_type)
print('Wikidata IDs:', sample.wikidata_ids[:5])
print('Num Context Paragraphs:', len(sample.context))
print('Num Evidence Triples:', len(sample.evidences))